# Experiments

Dieses Notebook enth?lt die systematische Fidelity/Sparsity-Evaluation f?r GNNExplainer und Integrated Gradients.

- Fidelity wird getrennt f?r `H` und `C` aggregiert.
- Standardm??ig wird nur der **erste Graph pro `compound`** im gew?hlten Scope verwendet.
- Scope kann zwischen `test_split` und `full_dataset` umgeschaltet werden.


In [1]:
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output


def _resolve_project_root() -> Path:
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "gnn4nmr",
        cwd.parent / "gnn4nmr",
    ]
    for candidate in candidates:
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError(
        f"Could not resolve project root from cwd={cwd}. Expected a folder containing scripts/ and notebooks/."
    )


PROJECT_ROOT = _resolve_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [str(PROJECT_ROOT), str(SCRIPTS_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"cwd: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Models dir: {MODELS_DIR}")


cwd: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\notebooks
Project root: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr
Notebook dir: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\notebooks
Data dir: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\data
Models dir: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\models


In [2]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

# Make this cell runnable on its own (even after kernel restart)
if "NOTEBOOK_DIR" not in globals():
    NOTEBOOK_DIR = Path.cwd()
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
if "MODELS_DIR" not in globals():
    MODELS_DIR = PROJECT_ROOT / "models"
if "DATA_DIR" not in globals():
    DATA_DIR = PROJECT_ROOT / "data"


def _list_files(directory: Path, extension: str):
    if directory.exists():
        return sorted([f.name for f in directory.glob(f"*{extension}")])
    return []


model_files = _list_files(MODELS_DIR, ".pt")
data_files = _list_files(DATA_DIR, ".pkl")

exp_model_widget = widgets.Dropdown(
    options=model_files if model_files else ["Keine Modelle gefunden"],
    description="Modell:",
    style={"description_width": "initial"},
)

exp_data_widget = widgets.Dropdown(
    options=data_files if data_files else ["Keine Daten gefunden"],
    description="Daten:",
    style={"description_width": "initial"},
)

split_file_widget = widgets.Text(
    value="models/graph_split.pkl",
    description="Split Datei:",
    style={"description_width": "initial"},
)

graph_scope_widget = widgets.Dropdown(
    options=[("Testsplit", "test_split"), ("Gesamtdatensatz", "full_dataset")],
    value="test_split",
    description="Graph Scope:",
    style={"description_width": "initial"},
)

first_graph_per_component_widget = widgets.Checkbox(
    value=True,
    description="Erster Graph pro Compound",
    indent=False,
)

component_key_widget = widgets.Text(
    value="compound",
    description="Component Key:",
    style={"description_width": "initial"},
)

node_types_widget = widgets.SelectMultiple(
    options=["H", "C", "Others"],
    value=("H", "C"),
    description="Node Types:",
    style={"description_width": "initial"},
)

sparsity_widget = widgets.FloatSlider(
    value=0.90,
    min=0.50,
    max=0.99,
    step=0.01,
    description="Sparsity:",
    style={"description_width": "initial"},
    readout_format=".2f",
)

mask_baseline_widget = widgets.Dropdown(
    options=["match_ig_baseline", "zero", "mean"],
    value="match_ig_baseline",
    description="Mask Baseline:",
    style={"description_width": "initial"},
)

ig_baseline_widget = widgets.Dropdown(
    options=["scientific", "zero", "mean", "random", "min", "max"],
    value="scientific",
    description="IG Baseline:",
    style={"description_width": "initial"},
)

gnn_epochs_widget = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description="GNN Epochs:",
    style={"description_width": "initial"},
)

gnn_lr_widget = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-4,
    max=-1,
    step=0.1,
    description="GNN LR:",
    style={"description_width": "initial"},
    readout_format=".4f",
)

gnn_explanation_type_widget = widgets.Dropdown(
    options=["phenomenon", "model"],
    value="phenomenon",
    description="GNN Type:",
    style={"description_width": "initial"},
)

ig_n_steps_widget = widgets.IntSlider(
    value=64,
    min=8,
    max=256,
    step=8,
    description="IG Steps:",
    style={"description_width": "initial"},
)

max_graphs_widget = widgets.IntText(
    value=0,
    description="Max Graphen (0=all):",
    style={"description_width": "initial"},
)

max_nodes_per_graph_widget = widgets.IntText(
    value=0,
    description="Max Nodes/Graph (0=all):",
    style={"description_width": "initial"},
)

include_edge_report_widget = widgets.Checkbox(
    value=True,
    description="GNN Edge Report",
    indent=False,
)

seed_widget = widgets.IntText(
    value=0,
    description="Seed:",
    style={"description_width": "initial"},
)

output_dir_widget = widgets.Text(
    value="results/experiments",
    description="Output Dir:",
    style={"description_width": "initial"},
)

controls = widgets.VBox(
    [
        widgets.HBox([exp_model_widget, exp_data_widget]),
        widgets.HBox([split_file_widget, graph_scope_widget]),
        widgets.HBox([first_graph_per_component_widget, component_key_widget]),
        widgets.HBox([node_types_widget, sparsity_widget]),
        widgets.HBox([mask_baseline_widget, ig_baseline_widget]),
        widgets.HBox([gnn_epochs_widget, gnn_lr_widget, gnn_explanation_type_widget]),
        widgets.HBox([ig_n_steps_widget, max_graphs_widget, max_nodes_per_graph_widget]),
        widgets.HBox([include_edge_report_widget, seed_widget, output_dir_widget]),
    ]
)

display(controls)


## Fidelity and Sparsity Metrics (Node-Level Regression)

Fuer jeden Knoten und jede Methode wird bei Ziel-Sparsity `s=0.90` zunaechst `k = max(1, ceil((1-s) * d))` (mit `d` = Feature-Anzahl) bestimmt.

- `S`: Top-k Features nach `|importance|`
- `actual_sparsity_feat = 1 - k/d`

Feature-Perturbationen am ausgewaehlten Knoten:

- `x_drop`: Features in `S` werden durch Baseline ersetzt
- `x_keep`: nur Features in `S` bleiben, Rest wird durch Baseline ersetzt

Mit `y_orig`, `y_drop`, `y_keep` und optional `y_true`:

- `fid_plus_model = |y_orig - y_drop|`
- `fid_minus_model = |y_orig - y_keep|`
- `fid_plus_error_delta = |y_drop - y_true| - |y_orig - y_true|`
- `fid_minus_error_delta = |y_keep - y_true| - |y_orig - y_true|`

Fuer den optionalen GNN-Edge-Report wird die Edge-Selektion global auf dem gesamten erklaerten Graphen durchgefuehrt:

- `E`: Anzahl aller beruecksichtigten Edges im Graphen
- `k_edge = max(1, ceil((1-s) * E))`, Ranking nach `|edge_mask|`
- Edge-`fid+`: Top-k Edges werden entfernt (`drop_selected`)
- Edge-`fid-`: Es bleiben nur die Top-k Edges erhalten (`keep_selected`)
- `actual_sparsity_edge = 1 - k_edge/E`

Die Aggregation erfolgt getrennt fuer `H` und `C`. Der faire Methodenvergleich nutzt nur die gemeinsame Node-Menge von GNNExplainer und IG.



In [3]:
from scripts.explainer.experiments_evaluation import (
    exp_load_eval_graph_indices,
    exp_build_mask_baseline,
    exp_topk_indices_from_importance,
    exp_predict_single_node,
    exp_feature_fidelity_for_node,
    exp_gnn_edge_fidelity_for_node,
    exp_extract_gnn_feature_importance,
    exp_extract_ig_feature_importance,
    select_scope_graph_indices,
    first_graph_indices_per_component,
    build_default_context,
    run_experiments_evaluation as _run_experiments_evaluation_core,
)


def run_experiments_evaluation(
    model_file=None,
    data_file=None,
    split_file=None,
    node_types=('H', 'C'),
    sparsity=0.90,
    mask_baseline_mode='match_ig_baseline',
    ig_baseline_mode='scientific',
    gnn_epochs=200,
    gnn_lr=0.01,
    gnn_explanation_type='phenomenon',
    ig_n_steps=64,
    graph_scope='test_split',
    first_graph_per_component=True,
    component_key='compound',
    max_graphs=None,
    max_nodes_per_graph=0,
    include_gnn_edge_report=True,
    output_dir='results/experiments',
    seed=0,
    verbose=True,
):
    model_name = model_file if model_file is not None else exp_model_widget.value
    data_name = data_file if data_file is not None else exp_data_widget.value

    if str(model_name).startswith('Keine') or str(data_name).startswith('Keine'):
        raise ValueError('Bitte g?ltige Modell- und Datendatei ausw?hlen.')

    split_eff = split_file if split_file is not None else split_file_widget.value

    context = build_default_context(
        project_root=PROJECT_ROOT,
        model_file=str(model_name),
        data_file=str(data_name),
        split_file=str(split_eff),
        output_dir=output_dir,
    )

    return _run_experiments_evaluation_core(
        context=context,
        node_types=node_types,
        sparsity=float(sparsity),
        mask_baseline_mode=str(mask_baseline_mode),
        ig_baseline_mode=str(ig_baseline_mode),
        gnn_epochs=int(gnn_epochs),
        gnn_lr=float(gnn_lr),
        gnn_explanation_type=str(gnn_explanation_type),
        gnn_use_custom_coeffs=False,
        gnn_coeffs=None,
        ig_n_steps=int(ig_n_steps),
        graph_scope=str(graph_scope),
        first_graph_per_component=bool(first_graph_per_component),
        component_key=str(component_key),
        max_graphs=max_graphs,
        max_nodes_per_graph=int(max_nodes_per_graph),
        include_gnn_edge_report=bool(include_gnn_edge_report),
        seed=int(seed),
        verbose=bool(verbose),
    )


In [4]:
run_button = widgets.Button(
    description='Experiments ausf?hren',
    button_style='success',
    icon='play'
)
run_output = widgets.Output()


def _run_clicked(_):
    with run_output:
        clear_output()
        try:
            global exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df
            max_graphs = int(max_graphs_widget.value)
            max_graphs = None if max_graphs <= 0 else max_graphs

            exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df = run_experiments_evaluation(
                node_types=tuple(node_types_widget.value),
                sparsity=float(sparsity_widget.value),
                mask_baseline_mode=mask_baseline_widget.value,
                ig_baseline_mode=ig_baseline_widget.value,
                gnn_epochs=int(gnn_epochs_widget.value),
                gnn_lr=float(gnn_lr_widget.value),
                gnn_explanation_type=gnn_explanation_type_widget.value,
                ig_n_steps=int(ig_n_steps_widget.value),
                graph_scope=graph_scope_widget.value,
                first_graph_per_component=bool(first_graph_per_component_widget.value),
                component_key=component_key_widget.value.strip() or 'compound',
                max_graphs=max_graphs,
                max_nodes_per_graph=int(max_nodes_per_graph_widget.value),
                include_gnn_edge_report=bool(include_edge_report_widget.value),
                output_dir=output_dir_widget.value,
                seed=int(seed_widget.value),
                verbose=True,
            )

            print('Summary by method/node type:')
            display(exp_summary_method_type_df)
            print('Fair comparison (shared nodes):')
            display(exp_summary_fair_df)
        except Exception as exc:
            print(f'? Fehler: {exc}')
            raise


run_button.on_click(_run_clicked)
display(run_button)
display(run_output)


Button(button_style='success', description='Experiments ausf?hren', icon='play', style=ButtonStyle())

Output()

In [5]:
# Optionaler Smoke-Run (bewusst auskommentiert):
# exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df = run_experiments_evaluation(
#     node_types=('H', 'C'),
#     sparsity=0.90,
#     mask_baseline_mode='match_ig_baseline',
#     ig_baseline_mode='scientific',
#     graph_scope='test_split',
#     first_graph_per_component=True,
#     component_key='compound',
#     max_graphs=2,
#     max_nodes_per_graph=5,
#     include_gnn_edge_report=True,
#     seed=0,
#     verbose=True,
# )
# display(exp_summary_method_type_df)
# display(exp_summary_fair_df)


### Hyperparameter Gridsearch für GNNExplainer

Diese Subsection fuehrt eine **staged Hyperparameter-Suche** fuer die vier GNNExplainer-Regularisierungen durch:

- **Stage A (2D):** `edge_size` x `edge_ent` bei neutraler Feature-Regularisierung
- **Stage B (2D):** `node_feat_size` x `node_feat_ent` bei neutraler Edge-Regularisierung
- **Stage C:** Kombination der besten `K` Kandidaten aus A und B (`K^2` Kombinationen)
- **Stage D (optional):** lokale Verfeinerung um das aktuell beste Set (eine log-Stufe nach unten/oben)

Die Suche verwendet **log-spaces** fuer die vier Koeffizienten, weil deren Einfluss typischerweise multiplikativ ist und auf mehreren Groessenordnungen stattfindet. Das ergibt stabilere und effizientere Raster als lineare Skalen.

Budget-Trennung:

- Suche auf `SEARCH_NODES` (schneller, konfigurierbar)
- Finale Bewertung auf `FINAL_NODES=150` Zielinstanzen


In [6]:
# Gridsearch Setup / Config
import time
import random
import itertools
import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from IPython.display import display
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import GNNExplainer
from torch_geometric.explain.config import (
    ModelConfig,
    ModelMode,
    ModelReturnType,
    ModelTaskLevel,
)

from scripts.explainer.experiments_evaluation import (
    exp_extract_gnn_feature_importance,
    exp_feature_fidelity_for_node,
    exp_gnn_edge_fidelity_for_node,
    first_graph_indices_per_component,
    get_train_graph_indices,
    select_scope_graph_indices,
    build_default_context,
)
from scripts.explainer.explainer_utils import (
    NodeTypeRegressionWrapper,
    build_dataset,
    get_device,
    heterodata_to_dicts,
    load_config,
    load_stats,
    load_trained_model,
)


def _safe_widget_value(widget_name, fallback):
    widget = globals().get(widget_name)
    if widget is None:
        return fallback
    try:
        value = widget.value
    except Exception:
        return fallback
    if isinstance(value, str) and value.startswith("Keine"):
        return fallback
    return value


# File defaults (widgets if available)
MODEL_FILE = str(_safe_widget_value("exp_model_widget", "SAGEConv_best_model.pt"))
DATA_FILE = str(_safe_widget_value("exp_data_widget", "all_graphs.pkl"))
SPLIT_FILE = str(_safe_widget_value("split_file_widget", "models/graph_split.pkl"))

# Main search budget
SEARCH_NODES = 30
FINAL_NODES = 150
SEEDS = [0, 1, 2]

# Explainer optimization
EPOCHS_SEARCH = 150
EPOCHS_FINAL = 250
LR = float(_safe_widget_value("gnn_lr_widget", 0.01))

# Stage control
TOP_K_STAGE = 3
STAGE_D_BUDGET = 0
STAGE_D_USE_SEARCH_EPOCHS = True

# Evaluation scope
NODE_TYPES = tuple(_safe_widget_value("node_types_widget", ("H", "C")))
GRAPH_SCOPE = str(_safe_widget_value("graph_scope_widget", "test_split"))
FIRST_GRAPH_PER_COMPONENT = bool(_safe_widget_value("first_graph_per_component_widget", True))
COMPONENT_KEY = str(_safe_widget_value("component_key_widget", "compound"))

# Fidelity config
SPARSITY = float(_safe_widget_value("sparsity_widget", 0.90))
MASK_BASELINE_MODE = str(_safe_widget_value("mask_baseline_widget", "match_ig_baseline"))
IG_BASELINE_MODE = str(_safe_widget_value("ig_baseline_widget", "scientific"))
GNN_EXPLANATION_TYPE = str(_safe_widget_value("gnn_explanation_type_widget", "phenomenon"))

# Selection reporting config (fixed but editable)
THRESHOLD_CONFIG = {
    "feature_threshold": 0.5,
    "edge_threshold": 0.5,
    "min_keep": 1,
    "use_abs": True,
}
TOPK_CONFIG = {
    "enabled": False,
    "topk_features": None,
    "topk_edges": None,
}

# Log-space grids
GRID_EDGE_SIZE = np.logspace(-6, -1, 6)
GRID_EDGE_ENT = np.logspace(-6, 0, 7)
GRID_NODE_FEAT_SIZE = np.logspace(-6, 0, 7)
GRID_NODE_FEAT_ENT = np.logspace(-6, -1, 6)

# Neutral regularization settings for staged search
NEUTRAL_EDGE_SIZE = 1e-12
NEUTRAL_EDGE_ENT = 1e-12
NEUTRAL_NODE_FEAT_SIZE = 1e-12
NEUTRAL_NODE_FEAT_ENT = 1e-12

print("Gridsearch config loaded.")
print(f"MODEL_FILE={MODEL_FILE}")
print(f"DATA_FILE={DATA_FILE}")
print(f"SPLIT_FILE={SPLIT_FILE}")
print(f"SEARCH_NODES={SEARCH_NODES}, FINAL_NODES={FINAL_NODES}, SEEDS={SEEDS}")



Gridsearch config loaded.
MODEL_FILE=GraphConv_best_model.pt
DATA_FILE=all_graphs_with_length.pkl
SPLIT_FILE=models/graph_split.pkl
SEARCH_NODES=30, FINAL_NODES=150, SEEDS=[0, 1, 2]


In [7]:
# Gridsearch helper functions

def set_all_seeds(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def build_gnnexplainer_algorithm(epochs, lr, coeffs):
    coeffs = dict(coeffs)
    try:
        return GNNExplainer(epochs=int(epochs), lr=float(lr), coeffs=coeffs)
    except TypeError:
        pass

    try:
        return GNNExplainer(epochs=int(epochs), lr=float(lr), **coeffs)
    except TypeError:
        pass

    algo = GNNExplainer(epochs=int(epochs), lr=float(lr))
    algo_coeffs = getattr(algo, "coeffs", None)
    if isinstance(algo_coeffs, dict):
        algo_coeffs.update(coeffs)
        return algo

    raise TypeError("Could not apply custom coeffs to this GNNExplainer version.")


def _clone_edge_attr_dict(edge_attr_dict):
    if edge_attr_dict is None:
        return None
    return {
        edge_type: (attrs.clone() if attrs is not None else None)
        for edge_type, attrs in edge_attr_dict.items()
    }


def _coeff_signature(coeffs):
    return (
        float(coeffs["edge_size"]),
        float(coeffs["edge_ent"]),
        float(coeffs["node_feat_size"]),
        float(coeffs["node_feat_ent"]),
    )


def load_eval_artifacts(
    project_root,
    model_file,
    data_file,
    split_file,
    node_types,
    graph_scope,
    first_graph_per_component,
    component_key,
):
    context = build_default_context(
        project_root=Path(project_root),
        model_file=str(model_file),
        data_file=str(data_file),
        split_file=str(split_file),
        output_dir="results/experiments",
    )

    device = get_device(None)
    config = load_config(str(context.config_path))
    norm_stats, edge_stats = load_stats(str(context.norm_stats_path), str(context.edge_stats_path))
    dataset = build_dataset(str(context.data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    base_model = load_trained_model(str(context.model_path), config, device)

    graph_indices = select_scope_graph_indices(dataset=dataset, split_path=context.split_path, graph_scope=graph_scope)
    if first_graph_per_component:
        graph_indices = first_graph_indices_per_component(
            dataset=dataset,
            graph_indices=graph_indices,
            component_key=component_key,
        )

    train_graph_indices = get_train_graph_indices(context.split_path, total_graphs=len(dataset))
    clean_node_types = tuple(nt for nt in node_types if nt in ("H", "C", "Others"))

    return {
        "context": context,
        "device": device,
        "config": config,
        "dataset": dataset,
        "base_model": base_model,
        "eval_graph_indices": [int(i) for i in graph_indices],
        "train_graph_indices": train_graph_indices,
        "node_types": clean_node_types,
    }


def collect_candidate_nodes(dataset, eval_graph_indices, node_types, explanation_type="phenomenon"):
    rows = []
    explanation_type = str(explanation_type)

    for graph_idx in sorted(int(i) for i in eval_graph_indices):
        data = dataset[graph_idx]

        for node_type in node_types:
            if node_type not in data.node_types:
                continue

            x = data[node_type].x
            num_nodes = int(x.size(0))
            if num_nodes <= 0:
                continue

            y_tensor = getattr(data[node_type], "y", None)
            y_flat = y_tensor.reshape(-1) if y_tensor is not None else None

            if explanation_type == "phenomenon" and y_flat is None:
                continue

            for node_idx in range(num_nodes):
                target_value = float("nan")
                if y_flat is not None and node_idx < int(y_flat.numel()):
                    value = y_flat[node_idx]
                    target_value = float(value.item())

                if explanation_type == "phenomenon" and not np.isfinite(target_value):
                    continue

                rows.append(
                    {
                        "graph_idx": int(graph_idx),
                        "node_type": str(node_type),
                        "node_idx": int(node_idx),
                        "target_value": float(target_value),
                    }
                )

    return rows


def sample_target_nodes(candidates, search_nodes, final_nodes, seed):
    if not candidates:
        raise ValueError("No candidate nodes available for search/evaluation.")

    rng = random.Random(int(seed))
    indices = list(range(len(candidates)))
    rng.shuffle(indices)
    shuffled = [candidates[i] for i in indices]

    if int(final_nodes) <= 0:
        final_count = len(shuffled)
    else:
        final_count = min(int(final_nodes), len(shuffled))

    final_selected = shuffled[:final_count]

    if int(search_nodes) <= 0:
        search_count = final_count
    else:
        search_count = min(int(search_nodes), final_count)

    search_selected = final_selected[:search_count]
    return search_selected, final_selected


def build_base_prediction_cache(base_model, dataset, target_nodes, device):
    graph_to_nodes = {}
    for row in target_nodes:
        graph_to_nodes.setdefault(int(row["graph_idx"]), []).append(row)

    graph_cache = {}
    pred_cache = {}

    for graph_idx in sorted(graph_to_nodes.keys()):
        data = dataset[graph_idx].to(device)
        x_dict_raw, edge_index_dict, edge_attr_dict_raw, y_dict = heterodata_to_dicts(data)

        x_dict = {nt: feat.clone() for nt, feat in x_dict_raw.items()}
        edge_attr_dict = _clone_edge_attr_dict(edge_attr_dict_raw)

        # The model forward mutates x_dict/edge_attr_dict in-place; use throwaway copies for pred cache.
        with torch.no_grad():
            pred_dict = base_model(
                {nt: feat.clone() for nt, feat in x_dict.items()},
                edge_index_dict,
                _clone_edge_attr_dict(edge_attr_dict),
            )

        graph_cache[graph_idx] = {
            "x_dict": x_dict,
            "edge_index_dict": edge_index_dict,
            "edge_attr_dict": edge_attr_dict,
            "y_dict": y_dict,
            "pred_dict": pred_dict,
        }

        for row in graph_to_nodes[graph_idx]:
            node_type = row["node_type"]
            node_idx = int(row["node_idx"])
            pred_tensor = pred_dict.get(node_type)
            if pred_tensor is None:
                continue
            flat = pred_tensor.reshape(-1)
            if 0 <= node_idx < int(flat.size(0)):
                pred_cache[(graph_idx, node_type, node_idx)] = float(flat[node_idx].item())

    return graph_cache, pred_cache


def count_selected_features(mask_values, threshold_cfg, topk_cfg):
    values = np.asarray(mask_values, dtype=float).reshape(-1)
    if values.size == 0:
        return 0

    use_abs = bool(threshold_cfg.get("use_abs", True))
    min_keep = max(0, int(threshold_cfg.get("min_keep", 0)))
    scores = np.abs(values) if use_abs else values

    if bool(topk_cfg.get("enabled", False)) and topk_cfg.get("topk_features") is not None:
        k = max(0, int(topk_cfg["topk_features"]))
        selected = min(k, int(scores.size))
        if min_keep > 0:
            selected = max(min_keep, selected)
        return int(min(selected, int(scores.size)))

    thr = float(threshold_cfg.get("feature_threshold", 0.5))
    selected = int(np.sum(np.isfinite(scores) & (scores >= thr)))
    if min_keep > 0:
        selected = max(min_keep, selected)
    return int(min(selected, int(scores.size)))


def count_selected_edges(edge_mask_dict, edge_index_dict, node_type, node_idx, threshold_cfg, topk_cfg):
    # Compatibility with existing call sites; edge counting is global across the graph.
    del node_type, node_idx

    use_abs = bool(threshold_cfg.get("use_abs", True))
    min_keep = max(0, int(threshold_cfg.get("min_keep", 0)))

    edge_scores = []
    for edge_type, edge_mask in edge_mask_dict.items():
        if edge_mask is None or edge_type not in edge_index_dict:
            continue

        edge_index = edge_index_dict[edge_type]
        mask_vals = edge_mask.view(-1).detach().cpu().numpy().reshape(-1)

        limit = min(mask_vals.shape[0], int(edge_index.size(1)))
        for pos in range(limit):
            score = float(mask_vals[pos])
            edge_scores.append(abs(score) if use_abs else score)

    if not edge_scores:
        return 0

    scores = np.asarray(edge_scores, dtype=float)

    if bool(topk_cfg.get("enabled", False)) and topk_cfg.get("topk_edges") is not None:
        k = max(0, int(topk_cfg["topk_edges"]))
        selected = min(k, int(scores.size))
        if min_keep > 0:
            selected = max(min_keep, selected)
        return int(min(selected, int(scores.size)))

    thr = float(threshold_cfg.get("edge_threshold", 0.5))
    selected = int(np.sum(np.isfinite(scores) & (scores >= thr)))
    if min_keep > 0:
        selected = max(min_keep, selected)
    return int(min(selected, int(scores.size)))


def _median(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.median(arr)) if arr.size > 0 else float("nan")


def _iqr(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan")
    q75, q25 = np.percentile(arr, [75, 25])
    return float(q75 - q25)


def evaluate_single_config(
    stage,
    coeffs,
    epochs,
    lr,
    seed,
    target_nodes,
    base_model,
    graph_cache,
    pred_cache,
    dataset,
    train_graph_indices,
    node_types,
    sparsity,
    mask_baseline_mode,
    ig_baseline_mode,
    explanation_type,
    threshold_cfg,
    topk_cfg,
    median_cache,
    elem_distribution_cache,
    baseline_cache_dir,
):
    set_all_seeds(seed)
    t0 = time.perf_counter()

    explainers = {}
    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )
    for node_type in node_types:
        wrapped_model = NodeTypeRegressionWrapper(base_model, node_type)
        algorithm = build_gnnexplainer_algorithm(epochs=epochs, lr=lr, coeffs=coeffs)
        explainers[node_type] = Explainer(
            model=wrapped_model,
            algorithm=algorithm,
            explanation_type=str(explanation_type),
            model_config=model_config,
            node_mask_type="attributes",
            edge_mask_type="object",
        )

    combined_vals = []
    abs_delta_vals = []
    edges_selected_vals = []
    feats_selected_vals = []
    failures = 0
    error_counts = {}
    fallback_model_target_count = 0
    error_first_message = ""

    for row in target_nodes:
        graph_idx = int(row["graph_idx"])
        node_type = str(row["node_type"])
        node_idx = int(row["node_idx"])
        target_value = float(row.get("target_value", float("nan")))

        if node_type not in explainers:
            continue
        if graph_idx not in graph_cache:
            failures += 1
            key = "graph_cache_missing"
            error_counts[key] = int(error_counts.get(key, 0)) + 1
            continue

        cache_item = graph_cache[graph_idx]
        x_dict_base = cache_item["x_dict"]
        edge_index_dict = cache_item["edge_index_dict"]
        edge_attr_base = cache_item["edge_attr_dict"]
        y_dict = cache_item["y_dict"]

        pred_orig = pred_cache.get((graph_idx, node_type, node_idx))
        if pred_orig is None:
            failures += 1
            key = "pred_orig_missing"
            error_counts[key] = int(error_counts.get(key, 0)) + 1
            continue

        gnn_target = None
        if str(explanation_type) == "phenomenon":
            y_tensor = y_dict.get(node_type)
            if y_tensor is None:
                failures += 1
                key = "target_missing"
                error_counts[key] = int(error_counts.get(key, 0)) + 1
                continue
            y_tensor = y_tensor.reshape(-1)
            if node_idx >= int(y_tensor.size(0)) or torch.isnan(y_tensor[node_idx]):
                failures += 1
                key = "target_nan_or_oob"
                error_counts[key] = int(error_counts.get(key, 0)) + 1
                continue
            gnn_target = y_tensor

        try:
            try:
                explanation = explainers[node_type](
                    {nt: feat.clone() for nt, feat in x_dict_base.items()},
                    edge_index_dict,
                    edge_attr_dict=_clone_edge_attr_dict(edge_attr_base),
                    target=gnn_target,
                    index=node_idx,
                )
            except Exception as exc_explain:
                # Compatibility fallback: indexed node-level calls can require scalar targets.
                if str(explanation_type) == "phenomenon" and gnn_target is not None:
                    try:
                        explanation = explainers[node_type](
                            {nt: feat.clone() for nt, feat in x_dict_base.items()},
                            edge_index_dict,
                            edge_attr_dict=_clone_edge_attr_dict(edge_attr_base),
                            target=gnn_target[node_idx],
                            index=node_idx,
                        )
                        fallback_model_target_count += 1
                    except Exception:
                        raise exc_explain
                else:
                    raise exc_explain

            feature_importance = exp_extract_gnn_feature_importance(
                explanation,
                node_type=node_type,
                node_idx=node_idx,
            )

            feature_metrics = exp_feature_fidelity_for_node(
                base_model=base_model,
                x_dict=x_dict_base,
                edge_index_dict=edge_index_dict,
                edge_attr_dict=edge_attr_base,
                node_type=node_type,
                node_idx=node_idx,
                importance=feature_importance,
                sparsity=float(sparsity),
                mask_baseline_mode=str(mask_baseline_mode),
                ig_baseline_mode=str(ig_baseline_mode),
                dataset=dataset,
                train_graph_indices=train_graph_indices,
                median_cache=median_cache,
                elem_distribution_cache=elem_distribution_cache,
                cache_dir=baseline_cache_dir,
                pred_orig=float(pred_orig),
                target_value=float(target_value),
            )

            edge_metrics = exp_gnn_edge_fidelity_for_node(
                base_model=base_model,
                x_dict=x_dict_base,
                edge_index_dict=edge_index_dict,
                edge_attr_dict=edge_attr_base,
                edge_mask_dict=explanation.edge_mask_dict,
                node_type=node_type,
                node_idx=node_idx,
                sparsity=float(sparsity),
                pred_orig=float(pred_orig),
                target_value=float(target_value),
            )

            feat_fid = float(feature_metrics.get("fid_plus_model", float("nan")))
            edge_fid = float(edge_metrics.get("fid_plus_model", float("nan"))) if edge_metrics is not None else float("nan")
            combined_fidelity = float(np.nanmean([feat_fid, edge_fid]))

            feat_delta = abs(float(feature_metrics.get("fid_plus_error_delta", float("nan"))))
            edge_delta = (
                abs(float(edge_metrics.get("fid_plus_error_delta", float("nan"))))
                if edge_metrics is not None
                else float("nan")
            )
            abs_delta = float(np.nanmean([feat_delta, edge_delta]))

            if not np.isfinite(combined_fidelity):
                failures += 1
                key = "nonfinite_combined_fidelity"
                error_counts[key] = int(error_counts.get(key, 0)) + 1
                continue

            feats_selected = count_selected_features(feature_importance, threshold_cfg=threshold_cfg, topk_cfg=topk_cfg)
            edges_selected = count_selected_edges(
                explanation.edge_mask_dict,
                edge_index_dict,
                node_type=node_type,
                node_idx=node_idx,
                threshold_cfg=threshold_cfg,
                topk_cfg=topk_cfg,
            )

            combined_vals.append(combined_fidelity)
            abs_delta_vals.append(abs_delta)
            feats_selected_vals.append(float(feats_selected))
            edges_selected_vals.append(float(edges_selected))
        except Exception as exc:
            failures += 1
            key = f"explain_failed:{type(exc).__name__}"
            error_counts[key] = int(error_counts.get(key, 0)) + 1
            if not error_first_message:
                error_first_message = f"{type(exc).__name__}: {exc}"

    runtime_sec = float(time.perf_counter() - t0)

    if error_counts:
        error_top_reason, error_top_count = sorted(error_counts.items(), key=lambda kv: (-int(kv[1]), str(kv[0])))[0]
    else:
        error_top_reason, error_top_count = "", 0

    row = {
        "stage": str(stage),
        "edge_size": float(coeffs["edge_size"]),
        "edge_ent": float(coeffs["edge_ent"]),
        "node_feat_size": float(coeffs["node_feat_size"]),
        "node_feat_ent": float(coeffs["node_feat_ent"]),
        "lr": float(lr),
        "epochs": int(epochs),
        "seed": int(seed),
        "fidelity_median": _median(combined_vals),
        "fidelity_iqr": _iqr(combined_vals),
        "abs_delta_median": _median(abs_delta_vals),
        "edges_selected_median": _median(edges_selected_vals),
        "feats_selected_median": _median(feats_selected_vals),
        "runtime_sec": runtime_sec,
        "n_nodes_evaluated": int(len(combined_vals)),
        "n_nodes_requested": int(len(target_nodes)),
        "n_failures": int(failures),
        "error_top_reason": str(error_top_reason),
        "error_top_count": int(error_top_count),
        "error_first_message": str(error_first_message),
        "fallback_model_target_count": int(fallback_model_target_count),
    }
    return row


def run_stage_grid(
    stage_name,
    coeff_dicts,
    epochs,
    seeds,
    target_nodes,
    base_model,
    graph_cache,
    pred_cache,
    dataset,
    train_graph_indices,
    node_types,
    sparsity,
    mask_baseline_mode,
    ig_baseline_mode,
    explanation_type,
    threshold_cfg,
    topk_cfg,
    median_cache,
    elem_distribution_cache,
    baseline_cache_dir,
    lr,
):
    rows = []
    for coeffs in coeff_dicts:
        for seed in seeds:
            rows.append(
                evaluate_single_config(
                    stage=stage_name,
                    coeffs=coeffs,
                    epochs=epochs,
                    lr=lr,
                    seed=seed,
                    target_nodes=target_nodes,
                    base_model=base_model,
                    graph_cache=graph_cache,
                    pred_cache=pred_cache,
                    dataset=dataset,
                    train_graph_indices=train_graph_indices,
                    node_types=node_types,
                    sparsity=sparsity,
                    mask_baseline_mode=mask_baseline_mode,
                    ig_baseline_mode=ig_baseline_mode,
                    explanation_type=explanation_type,
                    threshold_cfg=threshold_cfg,
                    topk_cfg=topk_cfg,
                    median_cache=median_cache,
                    elem_distribution_cache=elem_distribution_cache,
                    baseline_cache_dir=baseline_cache_dir,
                )
            )

    return pd.DataFrame(rows)


def aggregate_stage_scores(df_stage):
    if df_stage is None or df_stage.empty:
        return pd.DataFrame()

    group_cols = [
        "stage",
        "edge_size",
        "edge_ent",
        "node_feat_size",
        "node_feat_ent",
        "lr",
        "epochs",
    ]
    metric_cols = [
        "fidelity_median",
        "fidelity_iqr",
        "abs_delta_median",
        "edges_selected_median",
        "feats_selected_median",
        "runtime_sec",
        "n_nodes_evaluated",
        "n_nodes_requested",
        "n_failures",
    ]

    rows = []
    for keys, group_df in df_stage.groupby(group_cols, dropna=False):
        row = {col: val for col, val in zip(group_cols, keys)}
        for metric in metric_cols:
            values = pd.to_numeric(group_df[metric], errors="coerce").dropna()
            row[metric] = float(values.median()) if not values.empty else float("nan")

        fidelity_values = pd.to_numeric(group_df["fidelity_median"], errors="coerce").dropna()
        row["fidelity_std"] = float(fidelity_values.std(ddof=0)) if not fidelity_values.empty else float("nan")
        row["seed_runs"] = int(group_df["seed"].nunique())
        row["selection_sum"] = float(row["edges_selected_median"] + row["feats_selected_median"])
        rows.append(row)

    out = pd.DataFrame(rows)
    return out.sort_values(["stage", "fidelity_median"], ascending=[True, False]).reset_index(drop=True)


def select_top_k_configs(df_stage_agg, k=3):
    if df_stage_agg is None or df_stage_agg.empty:
        return pd.DataFrame()

    work = df_stage_agg.copy()
    if "selection_sum" not in work.columns:
        work["selection_sum"] = work["edges_selected_median"] + work["feats_selected_median"]

    work = work.sort_values(
        ["fidelity_median", "selection_sum", "runtime_sec"],
        ascending=[False, True, True],
    )
    return work.head(max(0, int(k))).reset_index(drop=True)


def build_stage_c_candidates(top_a_df, top_b_df):
    if top_a_df is None or top_b_df is None or top_a_df.empty or top_b_df.empty:
        return []

    candidates = []
    seen = set()
    for row_a in top_a_df.itertuples(index=False):
        for row_b in top_b_df.itertuples(index=False):
            coeffs = {
                "edge_size": float(row_a.edge_size),
                "edge_ent": float(row_a.edge_ent),
                "node_feat_size": float(row_b.node_feat_size),
                "node_feat_ent": float(row_b.node_feat_ent),
            }
            sig = _coeff_signature(coeffs)
            if sig in seen:
                continue
            seen.add(sig)
            candidates.append(coeffs)
    return candidates


def build_stage_d_candidates(best_cfg, grids, seen_cfgs, budget):
    budget = int(budget)
    if budget <= 0:
        return []

    def neighbor_values(grid_vals, current_val):
        grid_vals = np.asarray(grid_vals, dtype=float)
        cur = max(float(current_val), 1e-12)
        idx = int(np.argmin(np.abs(np.log10(grid_vals) - np.log10(cur))))
        idxs = [i for i in [idx - 1, idx, idx + 1] if 0 <= i < grid_vals.size]
        return sorted({float(grid_vals[i]) for i in idxs})

    edge_size_vals = neighbor_values(grids["edge_size"], best_cfg["edge_size"])
    edge_ent_vals = neighbor_values(grids["edge_ent"], best_cfg["edge_ent"])
    node_feat_size_vals = neighbor_values(grids["node_feat_size"], best_cfg["node_feat_size"])
    node_feat_ent_vals = neighbor_values(grids["node_feat_ent"], best_cfg["node_feat_ent"])

    candidates = []
    for es, ee, nfs, nfe in itertools.product(
        edge_size_vals,
        edge_ent_vals,
        node_feat_size_vals,
        node_feat_ent_vals,
    ):
        coeffs = {
            "edge_size": float(es),
            "edge_ent": float(ee),
            "node_feat_size": float(nfs),
            "node_feat_ent": float(nfe),
        }
        sig = _coeff_signature(coeffs)
        if sig in seen_cfgs:
            continue
        candidates.append(coeffs)

    candidates = sorted(
        candidates,
        key=lambda c: (c["edge_size"], c["edge_ent"], c["node_feat_size"], c["node_feat_ent"]),
    )
    return candidates[:budget]


def compute_pareto_front(df_agg):
    if df_agg is None or df_agg.empty:
        return pd.DataFrame()

    work = df_agg.copy()
    if "selection_sum" not in work.columns:
        work["selection_sum"] = work["edges_selected_median"] + work["feats_selected_median"]

    work = work[
        np.isfinite(pd.to_numeric(work["fidelity_median"], errors="coerce"))
        & np.isfinite(pd.to_numeric(work["selection_sum"], errors="coerce"))
    ].reset_index(drop=True)

    keep_indices = []
    for i, row_i in work.iterrows():
        dominated = False
        for j, row_j in work.iterrows():
            if i == j:
                continue

            better_or_equal = (
                float(row_j["fidelity_median"]) >= float(row_i["fidelity_median"])
                and float(row_j["selection_sum"]) <= float(row_i["selection_sum"])
            )
            strictly_better = (
                float(row_j["fidelity_median"]) > float(row_i["fidelity_median"])
                or float(row_j["selection_sum"]) < float(row_i["selection_sum"])
            )
            if better_or_equal and strictly_better:
                dominated = True
                break

        if not dominated:
            keep_indices.append(i)

    pareto = work.loc[keep_indices].copy()
    pareto = pareto.sort_values(["fidelity_median", "selection_sum"], ascending=[False, True]).reset_index(drop=True)
    return pareto







In [8]:
# Run staged gridsearch (A/B/C/(D)) + final evaluation
set_all_seeds(SEEDS[0])

artifacts = load_eval_artifacts(
    project_root=PROJECT_ROOT,
    model_file=MODEL_FILE,
    data_file=DATA_FILE,
    split_file=SPLIT_FILE,
    node_types=NODE_TYPES,
    graph_scope=GRAPH_SCOPE,
    first_graph_per_component=FIRST_GRAPH_PER_COMPONENT,
    component_key=COMPONENT_KEY,
)

dataset = artifacts["dataset"]
base_model = artifacts["base_model"]
device = artifacts["device"]
eval_graph_indices = artifacts["eval_graph_indices"]
train_graph_indices = artifacts["train_graph_indices"]
node_types_eff = artifacts["node_types"]
context = artifacts["context"]

candidate_nodes = collect_candidate_nodes(
    dataset=dataset,
    eval_graph_indices=eval_graph_indices,
    node_types=node_types_eff,
    explanation_type=GNN_EXPLANATION_TYPE,
)

if not candidate_nodes:
    raise RuntimeError("No valid candidate nodes found. Check scope/node types/targets.")

search_target_nodes, final_target_nodes = sample_target_nodes(
    candidates=candidate_nodes,
    search_nodes=SEARCH_NODES,
    final_nodes=FINAL_NODES,
    seed=SEEDS[0],
)

if len(final_target_nodes) < int(FINAL_NODES):
    print(
        f"Warning: only {len(final_target_nodes)} valid nodes available "
        f"(requested FINAL_NODES={FINAL_NODES})."
    )

print(f"Candidate nodes: {len(candidate_nodes)}")
print(f"Search nodes: {len(search_target_nodes)}")
print(f"Final nodes: {len(final_target_nodes)}")

# Cache graph data + base predictions for selected final nodes.
graph_cache, pred_cache = build_base_prediction_cache(
    base_model=base_model,
    dataset=dataset,
    target_nodes=final_target_nodes,
    device=device,
)

# Baseline caches for scientific baseline mode.
median_cache = {}
elem_distribution_cache = {}
if str(IG_BASELINE_MODE) == "scientific":
    try:
        from scripts.explainer.baselines import (
            load_or_compute_element_distribution,
            load_or_compute_medians,
        )

        baseline_cache_dir = context.output_dir.parent / "baselines"
        for nt in node_types_eff:
            try:
                median_cache[nt] = load_or_compute_medians(
                    dataset=dataset,
                    train_graph_indices=train_graph_indices,
                    node_type=nt,
                    cache_dir=str(baseline_cache_dir),
                )
            except Exception:
                pass
        try:
            elem_distribution_cache["value"] = load_or_compute_element_distribution(
                dataset=dataset,
                train_graph_indices=train_graph_indices,
                cache_dir=str(baseline_cache_dir),
            )
        except Exception:
            pass
    except Exception:
        baseline_cache_dir = context.output_dir.parent / "baselines"
else:
    baseline_cache_dir = context.output_dir.parent / "baselines"

# Stage A: edge regularization (node-feature regularization neutral)
stage_a_coeffs = [
    {
        "edge_size": float(edge_size),
        "edge_ent": float(edge_ent),
        "node_feat_size": float(NEUTRAL_NODE_FEAT_SIZE),
        "node_feat_ent": float(NEUTRAL_NODE_FEAT_ENT),
    }
    for edge_size, edge_ent in itertools.product(GRID_EDGE_SIZE, GRID_EDGE_ENT)
]

stage_a_df = run_stage_grid(
    stage_name="A",
    coeff_dicts=stage_a_coeffs,
    epochs=EPOCHS_SEARCH,
    seeds=SEEDS,
    target_nodes=search_target_nodes,
    base_model=base_model,
    graph_cache=graph_cache,
    pred_cache=pred_cache,
    dataset=dataset,
    train_graph_indices=train_graph_indices,
    node_types=node_types_eff,
    sparsity=SPARSITY,
    mask_baseline_mode=MASK_BASELINE_MODE,
    ig_baseline_mode=IG_BASELINE_MODE,
    explanation_type=GNN_EXPLANATION_TYPE,
    threshold_cfg=THRESHOLD_CONFIG,
    topk_cfg=TOPK_CONFIG,
    median_cache=median_cache,
    elem_distribution_cache=elem_distribution_cache,
    baseline_cache_dir=baseline_cache_dir,
    lr=LR,
)
stage_a_summary = aggregate_stage_scores(stage_a_df)
stage_a_top_df = select_top_k_configs(stage_a_summary, k=TOP_K_STAGE)

# Stage B: node-feature regularization (edge regularization neutral)
stage_b_coeffs = [
    {
        "edge_size": float(NEUTRAL_EDGE_SIZE),
        "edge_ent": float(NEUTRAL_EDGE_ENT),
        "node_feat_size": float(node_feat_size),
        "node_feat_ent": float(node_feat_ent),
    }
    for node_feat_size, node_feat_ent in itertools.product(GRID_NODE_FEAT_SIZE, GRID_NODE_FEAT_ENT)
]

stage_b_df = run_stage_grid(
    stage_name="B",
    coeff_dicts=stage_b_coeffs,
    epochs=EPOCHS_SEARCH,
    seeds=SEEDS,
    target_nodes=search_target_nodes,
    base_model=base_model,
    graph_cache=graph_cache,
    pred_cache=pred_cache,
    dataset=dataset,
    train_graph_indices=train_graph_indices,
    node_types=node_types_eff,
    sparsity=SPARSITY,
    mask_baseline_mode=MASK_BASELINE_MODE,
    ig_baseline_mode=IG_BASELINE_MODE,
    explanation_type=GNN_EXPLANATION_TYPE,
    threshold_cfg=THRESHOLD_CONFIG,
    topk_cfg=TOPK_CONFIG,
    median_cache=median_cache,
    elem_distribution_cache=elem_distribution_cache,
    baseline_cache_dir=baseline_cache_dir,
    lr=LR,
)
stage_b_summary = aggregate_stage_scores(stage_b_df)
stage_b_top_df = select_top_k_configs(stage_b_summary, k=TOP_K_STAGE)

# Stage C: combine top-K from A and B
stage_c_coeffs = build_stage_c_candidates(stage_a_top_df, stage_b_top_df)
if stage_c_coeffs:
    stage_c_df = run_stage_grid(
        stage_name="C",
        coeff_dicts=stage_c_coeffs,
        epochs=EPOCHS_SEARCH,
        seeds=SEEDS,
        target_nodes=search_target_nodes,
        base_model=base_model,
        graph_cache=graph_cache,
        pred_cache=pred_cache,
        dataset=dataset,
        train_graph_indices=train_graph_indices,
        node_types=node_types_eff,
        sparsity=SPARSITY,
        mask_baseline_mode=MASK_BASELINE_MODE,
        ig_baseline_mode=IG_BASELINE_MODE,
        explanation_type=GNN_EXPLANATION_TYPE,
        threshold_cfg=THRESHOLD_CONFIG,
        topk_cfg=TOPK_CONFIG,
        median_cache=median_cache,
        elem_distribution_cache=elem_distribution_cache,
        baseline_cache_dir=baseline_cache_dir,
        lr=LR,
    )
    stage_c_summary = aggregate_stage_scores(stage_c_df)
else:
    stage_c_df = pd.DataFrame()
    stage_c_summary = pd.DataFrame()

raw_parts = [df for df in [stage_a_df, stage_b_df, stage_c_df] if not df.empty]
summary_parts = [df for df in [stage_a_summary, stage_b_summary, stage_c_summary] if not df.empty]

search_summary_df = pd.concat(summary_parts, ignore_index=True) if summary_parts else pd.DataFrame()

# Stage D: optional local refinement around current best config
if STAGE_D_BUDGET > 0 and not search_summary_df.empty:
    best_pre_df = select_top_k_configs(search_summary_df, k=1)
    best_pre = best_pre_df.iloc[0].to_dict()

    seen_cfgs = {
        _coeff_signature(
            {
                "edge_size": row.edge_size,
                "edge_ent": row.edge_ent,
                "node_feat_size": row.node_feat_size,
                "node_feat_ent": row.node_feat_ent,
            }
        )
        for row in search_summary_df.itertuples(index=False)
    }

    stage_d_coeffs = build_stage_d_candidates(
        best_cfg=best_pre,
        grids={
            "edge_size": GRID_EDGE_SIZE,
            "edge_ent": GRID_EDGE_ENT,
            "node_feat_size": GRID_NODE_FEAT_SIZE,
            "node_feat_ent": GRID_NODE_FEAT_ENT,
        },
        seen_cfgs=seen_cfgs,
        budget=STAGE_D_BUDGET,
    )

    if stage_d_coeffs:
        stage_d_epochs = EPOCHS_SEARCH if STAGE_D_USE_SEARCH_EPOCHS else EPOCHS_FINAL
        stage_d_df = run_stage_grid(
            stage_name="D",
            coeff_dicts=stage_d_coeffs,
            epochs=stage_d_epochs,
            seeds=SEEDS,
            target_nodes=search_target_nodes,
            base_model=base_model,
            graph_cache=graph_cache,
            pred_cache=pred_cache,
            dataset=dataset,
            train_graph_indices=train_graph_indices,
            node_types=node_types_eff,
            sparsity=SPARSITY,
            mask_baseline_mode=MASK_BASELINE_MODE,
            ig_baseline_mode=IG_BASELINE_MODE,
            explanation_type=GNN_EXPLANATION_TYPE,
            threshold_cfg=THRESHOLD_CONFIG,
            topk_cfg=TOPK_CONFIG,
            median_cache=median_cache,
            elem_distribution_cache=elem_distribution_cache,
            baseline_cache_dir=baseline_cache_dir,
            lr=LR,
        )
        stage_d_summary = aggregate_stage_scores(stage_d_df)
        stage_d_top_df = select_top_k_configs(stage_d_summary, k=min(TOP_K_STAGE, max(1, STAGE_D_BUDGET)))

        raw_parts.append(stage_d_df)
        summary_parts.append(stage_d_summary)
    else:
        stage_d_df = pd.DataFrame()
        stage_d_summary = pd.DataFrame()
        stage_d_top_df = pd.DataFrame()
else:
    stage_d_df = pd.DataFrame()
    stage_d_summary = pd.DataFrame()
    stage_d_top_df = pd.DataFrame()

# Combined search summary and ranking
stage_summary_df = pd.concat(summary_parts, ignore_index=True) if summary_parts else pd.DataFrame()
search_summary_df = stage_summary_df[stage_summary_df["stage"].isin(["A", "B", "C", "D"])].copy() if not stage_summary_df.empty else pd.DataFrame()

if search_summary_df.empty:
    raise RuntimeError("Search stages produced no valid results.")

search_summary_df["fidelity_median"] = pd.to_numeric(search_summary_df["fidelity_median"], errors="coerce")
if not np.isfinite(search_summary_df["fidelity_median"].to_numpy()).any():
    diag = pd.concat(raw_parts, ignore_index=True) if raw_parts else pd.DataFrame()
    for c in ["fidelity_median", "n_nodes_evaluated", "n_nodes_requested", "n_failures", "error_top_count"]:
        if c in diag.columns:
            diag[c] = pd.to_numeric(diag[c], errors="coerce")
    if not diag.empty:
        diag_summary = (
            diag.groupby("stage", dropna=False)
            .agg(
                rows=("stage", "size"),
                finite_fidelity=("fidelity_median", lambda s: int(np.isfinite(s).sum())),
                median_nodes_eval=("n_nodes_evaluated", "median"),
                median_nodes_req=("n_nodes_requested", "median"),
                median_failures=("n_failures", "median"),
                median_error_top_count=("error_top_count", "median"),
            )
            .reset_index()
        )
        if "error_top_reason" in diag.columns:
            reason_summary = (
                diag[diag["error_top_reason"].astype(str) != ""]
                .groupby(["stage", "error_top_reason"], dropna=False)
                .size()
                .reset_index(name="rows")
                .sort_values(["stage", "rows"], ascending=[True, False])
            )
        else:
            reason_summary = pd.DataFrame()

        if "error_first_message" in diag.columns:
            msg_summary = (
                diag[diag["error_first_message"].astype(str) != ""]
                .groupby(["stage", "error_first_message"], dropna=False)
                .size()
                .reset_index(name="rows")
                .sort_values(["stage", "rows"], ascending=[True, False])
            )
        else:
            msg_summary = pd.DataFrame()

        print("Search diagnostics by stage:")
        display(diag_summary)
        if not reason_summary.empty:
            print("Top error reasons:")
            display(reason_summary.groupby("stage", dropna=False).head(5))
        if not msg_summary.empty:
            print("Top error messages:")
            display(msg_summary.groupby("stage", dropna=False).head(3))
    raise RuntimeError(
        "Search produced no finite fidelities. Check diagnostics; likely all explainer runs failed for selected nodes/config."
    )


best_search_df = select_top_k_configs(search_summary_df, k=1)
best_search_row = best_search_df.iloc[0]

best_config = {
    "edge_size": float(best_search_row["edge_size"]),
    "edge_ent": float(best_search_row["edge_ent"]),
    "node_feat_size": float(best_search_row["node_feat_size"]),
    "node_feat_ent": float(best_search_row["node_feat_ent"]),
    "lr": float(LR),
    "epochs": int(EPOCHS_FINAL),
}

# Final evaluation on FINAL_NODES with best config
final_coeffs = [
    {
        "edge_size": best_config["edge_size"],
        "edge_ent": best_config["edge_ent"],
        "node_feat_size": best_config["node_feat_size"],
        "node_feat_ent": best_config["node_feat_ent"],
    }
]

final_results_df = run_stage_grid(
    stage_name="FINAL",
    coeff_dicts=final_coeffs,
    epochs=EPOCHS_FINAL,
    seeds=SEEDS,
    target_nodes=final_target_nodes,
    base_model=base_model,
    graph_cache=graph_cache,
    pred_cache=pred_cache,
    dataset=dataset,
    train_graph_indices=train_graph_indices,
    node_types=node_types_eff,
    sparsity=SPARSITY,
    mask_baseline_mode=MASK_BASELINE_MODE,
    ig_baseline_mode=IG_BASELINE_MODE,
    explanation_type=GNN_EXPLANATION_TYPE,
    threshold_cfg=THRESHOLD_CONFIG,
    topk_cfg=TOPK_CONFIG,
    median_cache=median_cache,
    elem_distribution_cache=elem_distribution_cache,
    baseline_cache_dir=baseline_cache_dir,
    lr=LR,
)
final_summary_df = aggregate_stage_scores(final_results_df)

# Full outputs
grid_results_df = pd.concat(raw_parts + ([final_results_df] if not final_results_df.empty else []), ignore_index=True)
stage_summary_df = pd.concat([stage_summary_df, final_summary_df], ignore_index=True) if not final_summary_df.empty else stage_summary_df

pareto_df = compute_pareto_front(search_summary_df)

required_cols = [
    "stage",
    "edge_size",
    "edge_ent",
    "node_feat_size",
    "node_feat_ent",
    "lr",
    "epochs",
    "seed",
    "fidelity_median",
    "fidelity_iqr",
    "abs_delta_median",
    "edges_selected_median",
    "feats_selected_median",
    "runtime_sec",
]
for col in required_cols:
    if col not in grid_results_df.columns:
        grid_results_df[col] = np.nan

print("Best config (search -> final):")
print(best_config)
print("\nTop Stage A candidates:")
display(stage_a_top_df)
print("Top Stage B candidates:")
display(stage_b_top_df)
if not stage_c_summary.empty:
    print("Top Stage C candidates:")
    display(select_top_k_configs(stage_c_summary, k=TOP_K_STAGE))
if not stage_d_summary.empty:
    print("Top Stage D candidates:")
    display(stage_d_top_df)

print("\nFinal summary:")
display(final_summary_df)

if not pareto_df.empty:
    print("Pareto front (search stages):")
    display(pareto_df.head(15))

print("grid_results_df columns:")
print(list(grid_results_df.columns))







c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\scripts\explainer\explainer_utils.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(model_path, map_location=device)


Candidate nodes: 201
Search nodes: 30
Final nodes: 150
Best config (search -> final):
{'edge_size': 1e-06, 'edge_ent': 1e-06, 'node_feat_size': 1e-12, 'node_feat_ent': 1e-12, 'lr': 0.01, 'epochs': 250}

Top Stage A candidates:


,stage,edge_size,edge_ent,node_feat_size,node_feat_ent,lr,epochs,fidelity_median,fidelity_iqr,abs_delta_median,edges_selected_median,feats_selected_median,runtime_sec,n_nodes_evaluated,n_nodes_requested,n_failures,fidelity_std,seed_runs,selection_sum
0,A,0.000001,0.000001,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,35.018104,30.0,30.0,0.0,0.001016,3,7.0
1,A,0.010000,1.000000,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,35.898455,30.0,30.0,0.0,0.001016,3,7.0
2,A,0.100000,0.010000,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,35.957689,30.0,30.0,0.0,0.001016,3,7.0


Top Stage B candidates:


,stage,edge_size,edge_ent,node_feat_size,node_feat_ent,lr,epochs,fidelity_median,fidelity_iqr,abs_delta_median,edges_selected_median,feats_selected_median,runtime_sec,n_nodes_evaluated,n_nodes_requested,n_failures,fidelity_std,seed_runs,selection_sum
0,B,1.000000e-12,1.000000e-12,0.000001,0.000001,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,35.919338,30.0,30.0,0.0,0.001016,3,7.0
1,B,1.000000e-12,1.000000e-12,0.000100,0.100000,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,35.960516,30.0,30.0,0.0,0.001016,3,7.0
2,B,1.000000e-12,1.000000e-12,0.001000,0.000100,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,36.018038,30.0,30.0,0.0,0.001016,3,7.0


...


Final summary:


,stage,edge_size,edge_ent,node_feat_size,node_feat_ent,lr,epochs,fidelity_median,fidelity_iqr,abs_delta_median,edges_selected_median,feats_selected_median,runtime_sec,n_nodes_evaluated,n_nodes_requested,n_failures,fidelity_std,seed_runs,selection_sum
0,FINAL,0.000001,0.000001,1.000000e-12,1.000000e-12,0.01,250,0.032797,0.72272,0.027717,6.5,1.0,325.447992,150.0,150.0,0.0,0.001899,3,7.5


Pareto front (search stages):


,stage,edge_size,edge_ent,node_feat_size,node_feat_ent,lr,epochs,fidelity_median,fidelity_iqr,abs_delta_median,edges_selected_median,feats_selected_median,runtime_sec,n_nodes_evaluated,n_nodes_requested,n_failures,fidelity_std,seed_runs,selection_sum
0,A,0.000001,0.000001,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,35.018104,30.0,30.0,0.0,0.001016,3,7.0
1,A,0.000001,0.000010,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,36.105059,30.0,30.0,0.0,0.001016,3,7.0
2,A,0.000001,0.000100,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,41.930335,30.0,30.0,0.0,0.001016,3,7.0
3,A,0.000001,0.001000,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,40.954294,30.0,30.0,0.0,0.001016,3,7.0
4,A,0.000001,0.010000,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,40.423029,30.0,30.0,0.0,0.001016,3,7.0
5,A,0.000001,0.100000,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,37.605719,30.0,30.0,0.0,0.001016,3,7.0
6,A,0.000001,1.000000,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,37.707611,30.0,30.0,0.0,0.001016,3,7.0
7,A,0.000010,0.000001,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,37.346349,30.0,30.0,0.0,0.001016,3,7.0
8,A,0.000010,0.000010,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,37.826545,30.0,30.0,0.0,0.001016,3,7.0
9,A,0.000010,0.000100,1.000000e-12,1.000000e-12,0.01,150,0.028959,0.41787,0.025138,6.0,1.0,38.062150,30.0,30.0,0.0,0.001016,3,7.0


grid_results_df columns:
['stage', 'edge_size', 'edge_ent', 'node_feat_size', 'node_feat_ent', 'lr', 'epochs', 'seed', 'fidelity_median', 'fidelity_iqr', 'abs_delta_median', 'edges_selected_median', 'feats_selected_median', 'runtime_sec', 'n_nodes_evaluated', 'n_nodes_requested', 'n_failures', 'error_top_reason', 'error_top_count', 'error_first_message', 'fallback_model_target_count']


In [9]:
# Plotting: parallel coordinates + tradeoff scatters


def _build_plot_df_from_seed_rows(seed_df):
    if seed_df is None or seed_df.empty:
        return pd.DataFrame()

    group_cols = [
        "stage",
        "edge_size",
        "edge_ent",
        "node_feat_size",
        "node_feat_ent",
        "lr",
        "epochs",
    ]
    metric_cols = [
        "fidelity_median",
        "edges_selected_median",
        "feats_selected_median",
        "runtime_sec",
        "n_nodes_evaluated",
        "n_nodes_requested",
        "n_failures",
    ]

    rows = []
    for keys, gdf in seed_df.groupby(group_cols, dropna=False):
        row = {col: val for col, val in zip(group_cols, keys)}
        for metric in metric_cols:
            vals = pd.to_numeric(gdf[metric], errors="coerce").dropna()
            row[metric] = float(vals.median()) if not vals.empty else float("nan")
        rows.append(row)

    return pd.DataFrame(rows)


plot_df = pd.DataFrame()

# Preferred source: aggregated stage summary
if "stage_summary_df" in globals() and isinstance(stage_summary_df, pd.DataFrame) and not stage_summary_df.empty:
    plot_df = stage_summary_df.copy()

# Fallback: aggregate from seed-level rows
if plot_df.empty and "grid_results_df" in globals() and isinstance(grid_results_df, pd.DataFrame) and not grid_results_df.empty:
    plot_df = _build_plot_df_from_seed_rows(grid_results_df.copy())

if plot_df.empty:
    raise RuntimeError("No plot data found. Run the staged gridsearch cell first.")

plot_df["fidelity_median"] = pd.to_numeric(plot_df["fidelity_median"], errors="coerce")
finite_mask = np.isfinite(plot_df["fidelity_median"].to_numpy())
plot_df = plot_df.loc[finite_mask].copy()

if plot_df.empty:
    print("No finite fidelity values found for plotting.")

    if "grid_results_df" in globals() and isinstance(grid_results_df, pd.DataFrame) and not grid_results_df.empty:
        diag = grid_results_df.copy()
        for c in ["fidelity_median", "n_nodes_evaluated", "n_nodes_requested", "n_failures"]:
            if c in diag.columns:
                diag[c] = pd.to_numeric(diag[c], errors="coerce")

        diag_summary = (
            diag.groupby("stage", dropna=False)
            .agg(
                rows=("stage", "size"),
                finite_fidelity=("fidelity_median", lambda s: int(np.isfinite(s).sum())),
                median_nodes_eval=("n_nodes_evaluated", "median"),
                median_nodes_req=("n_nodes_requested", "median"),
                median_failures=("n_failures", "median"),
            )
            .reset_index()
        )
        print("Diagnostic summary by stage:")
        display(diag_summary)

    raise RuntimeError(
        "No finite values available for plotting. Inspect diagnostic summary above and rerun search cell."
    )

for col in ["edge_size", "edge_ent", "node_feat_size", "node_feat_ent"]:
    vals = pd.to_numeric(plot_df[col], errors="coerce").astype(float)
    plot_df[f"log10_{col}"] = np.log10(np.clip(vals, 1e-12, None))

parallel_dims = [
    "log10_edge_size",
    "log10_edge_ent",
    "log10_node_feat_size",
    "log10_node_feat_ent",
    "fidelity_median",
    "edges_selected_median",
    "feats_selected_median",
]

fig_parallel = px.parallel_coordinates(
    plot_df,
    dimensions=parallel_dims,
    color="fidelity_median",
    color_continuous_scale=px.colors.sequential.Viridis,
    labels={
        "log10_edge_size": "log10(edge_size)",
        "log10_edge_ent": "log10(edge_ent)",
        "log10_node_feat_size": "log10(node_feat_size)",
        "log10_node_feat_ent": "log10(node_feat_ent)",
        "fidelity_median": "fidelity_median",
        "edges_selected_median": "edges_selected_median",
        "feats_selected_median": "feats_selected_median",
    },
)
fig_parallel.update_layout(title="GNNExplainer Hyperparameter Gridsearch - Parallel Coordinates")
fig_parallel.show()

fig_edges = px.scatter(
    plot_df,
    x="edges_selected_median",
    y="fidelity_median",
    color="stage",
    hover_data=["edge_size", "edge_ent", "node_feat_size", "node_feat_ent", "runtime_sec"],
    title="Tradeoff: fidelity_median vs edges_selected_median",
)
fig_edges.show()

fig_feats = px.scatter(
    plot_df,
    x="feats_selected_median",
    y="fidelity_median",
    color="stage",
    hover_data=["edge_size", "edge_ent", "node_feat_size", "node_feat_ent", "runtime_sec"],
    title="Tradeoff: fidelity_median vs feats_selected_median",
)
fig_feats.show()


In [10]:
# ── Plotting: parallel coordinates + tradeoff scatters ────────────────────────

def _build_plot_df_from_seed_rows(seed_df):
    if seed_df is None or seed_df.empty:
        return pd.DataFrame()

    group_cols = ["stage", "edge_size", "edge_ent", "node_feat_size",
                  "node_feat_ent", "lr", "epochs"]
    metric_cols = ["fidelity_median", "edges_selected_median",
                   "feats_selected_median", "runtime_sec",
                   "n_nodes_evaluated", "n_nodes_requested", "n_failures"]

    rows = []
    for keys, gdf in seed_df.groupby(group_cols, dropna=False):
        row = {col: val for col, val in zip(group_cols, keys)}
        for metric in metric_cols:
            vals = pd.to_numeric(gdf[metric], errors="coerce").dropna()
            row[metric] = float(vals.median()) if not vals.empty else float("nan")
        rows.append(row)
    return pd.DataFrame(rows)


# ── Data preparation ───────────────────────────────────────────────────────────
plot_df = pd.DataFrame()

if "stage_summary_df" in globals() and isinstance(stage_summary_df, pd.DataFrame) and not stage_summary_df.empty:
    plot_df = stage_summary_df.copy()

if plot_df.empty and "grid_results_df" in globals() and isinstance(grid_results_df, pd.DataFrame) and not grid_results_df.empty:
    plot_df = _build_plot_df_from_seed_rows(grid_results_df.copy())

if plot_df.empty:
    raise RuntimeError("No plot data found. Run the staged gridsearch cell first.")

plot_df["fidelity_median"] = pd.to_numeric(plot_df["fidelity_median"], errors="coerce")
finite_mask = np.isfinite(plot_df["fidelity_median"].to_numpy())
plot_df = plot_df.loc[finite_mask].copy()

if plot_df.empty:
    if "grid_results_df" in globals() and isinstance(grid_results_df, pd.DataFrame) and not grid_results_df.empty:
        diag = grid_results_df.copy()
        for c in ["fidelity_median", "n_nodes_evaluated", "n_nodes_requested", "n_failures"]:
            if c in diag.columns:
                diag[c] = pd.to_numeric(diag[c], errors="coerce")
        diag_summary = (
            diag.groupby("stage", dropna=False)
            .agg(rows=("stage", "size"),
                 finite_fidelity=("fidelity_median", lambda s: int(np.isfinite(s).sum())),
                 median_nodes_eval=("n_nodes_evaluated", "median"),
                 median_nodes_req=("n_nodes_requested", "median"),
                 median_failures=("n_failures", "median"))
            .reset_index()
        )
        print("Diagnostic summary by stage:")
        display(diag_summary)
    raise RuntimeError(
        "No finite values available for plotting. Inspect diagnostic summary above and rerun search cell."
    )

for col in ["edge_size", "edge_ent", "node_feat_size", "node_feat_ent"]:
    vals = pd.to_numeric(plot_df[col], errors="coerce").astype(float)
    plot_df[f"log10_{col}"] = np.log10(np.clip(vals, 1e-12, None))


# ── 1. Parallel Coordinates ───────────────────────────────────────────────────
parallel_dims = [
    "log10_edge_size", "log10_edge_ent",
    "log10_node_feat_size", "log10_node_feat_ent",
    "fidelity_median", "edges_selected_median", "feats_selected_median",
]

# Short labels avoid overlapping axis titles
short_labels = {
    "log10_edge_size":       "log₁₀<br>edge_size",
    "log10_edge_ent":        "log₁₀<br>edge_ent",
    "log10_node_feat_size":  "log₁₀<br>nf_size",
    "log10_node_feat_ent":   "log₁₀<br>nf_ent",
    "fidelity_median":       "fidelity<br>median",
    "edges_selected_median": "edges<br>selected",
    "feats_selected_median": "feats<br>selected",
}

fig_parallel = px.parallel_coordinates(
    plot_df,
    dimensions=parallel_dims,
    color="fidelity_median",
    color_continuous_scale=px.colors.sequential.Plasma,
    labels=short_labels,
)

fig_parallel.update_layout(
    title=dict(
        text="GNNExplainer Hyperparameter Gridsearch — Parallel Coordinates",
        font=dict(size=16, family="Arial"),
        x=0.5, xanchor="center",
    ),
    width=1300,
    height=550,
    margin=dict(l=80, r=150, t=100, b=80),   # extra top/bottom for labels
    font=dict(size=12, family="Arial"),
    coloraxis_colorbar=dict(
        title=dict(text="fidelity<br>median", font=dict(size=12)),
        thickness=18,
        len=0.75,
        yanchor="middle", y=0.5,
        tickfont=dict(size=11),
    ),
    paper_bgcolor="#1e1e2e",
    plot_bgcolor="#1e1e2e",
    font_color="white",
)

# Make axis labels sit *above* the axis (labelangle prevents overlap)
for dim in fig_parallel.data[0].dimensions:
    dim.label = short_labels.get(dim.label, dim.label)

fig_parallel.show()


# ── 2 & 3. Tradeoff Scatterplots (side by side) ───────────────────────────────
from plotly.subplots import make_subplots
import plotly.graph_objects as go

stage_colors = px.colors.qualitative.Bold   # distinct palette per stage
stages       = sorted(plot_df["stage"].dropna().unique())
color_map    = {s: stage_colors[i % len(stage_colors)] for i, s in enumerate(stages)}

hover_cols = ["edge_size", "edge_ent", "node_feat_size", "node_feat_ent", "runtime_sec"]

fig_trade = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Fidelity vs Edges Selected",
        "Fidelity vs Feats Selected",
    ],
    horizontal_spacing=0.12,
)

for x_col, col_idx in [("edges_selected_median", 1), ("feats_selected_median", 2)]:
    for stage in stages:
        sub = plot_df[plot_df["stage"] == stage]

        hover_text = sub.apply(
            lambda r: (
                f"<b>Stage {r['stage']}</b><br>"
                f"edge_size: {r['edge_size']}<br>"
                f"edge_ent: {r['edge_ent']}<br>"
                f"node_feat_size: {r['node_feat_size']}<br>"
                f"node_feat_ent: {r['node_feat_ent']}<br>"
                f"runtime: {r['runtime_sec']:.1f}s"
            ),
            axis=1,
        )

        fig_trade.add_trace(
            go.Scatter(
                x=sub[x_col],
                y=sub["fidelity_median"],
                mode="markers",
                name=f"Stage {stage}",
                legendgroup=f"stage_{stage}",
                showlegend=(col_idx == 1),          # legend only once
                marker=dict(
                    color=color_map[stage],
                    size=9,
                    opacity=0.82,
                    line=dict(width=0.8, color="white"),
                ),
                hovertemplate=hover_text + "<extra></extra>",
            ),
            row=1, col=col_idx,
        )

x_labels = {1: "Edges Selected (median)", 2: "Feats Selected (median)"}
for col_idx in [1, 2]:
    fig_trade.update_xaxes(
        title_text=x_labels[col_idx],
        title_font=dict(size=13),
        gridcolor="#3a3a4a",
        zeroline=False,
        row=1, col=col_idx,
    )
    fig_trade.update_yaxes(
        title_text="Fidelity (median)" if col_idx == 1 else "",
        title_font=dict(size=13),
        gridcolor="#3a3a4a",
        zeroline=False,
        row=1, col=col_idx,
    )

fig_trade.update_layout(
    title=dict(
        text="GNNExplainer — Fidelity Tradeoff by Stage",
        font=dict(size=16, family="Arial"),
        x=0.5, xanchor="center",
    ),
    width=1200,
    height=480,
    margin=dict(l=70, r=40, t=80, b=60),
    paper_bgcolor="#1e1e2e",
    plot_bgcolor="#1e1e2e",
    font=dict(color="white", size=12, family="Arial"),
    legend=dict(
        title="Stage",
        bgcolor="rgba(255,255,255,0.08)",
        bordercolor="rgba(255,255,255,0.2)",
        borderwidth=1,
        font=dict(size=12),
    ),
)

fig_trade.show()